In [ ]:
import sys
from collections import defaultdict

gtf_file = "hg38.ncbiRefSeq.gtf"
output_file = "optional_exons.bed"

gene_transcripts = defaultdict(set)

exon_transcripts = defaultdict(set)

def parse_attributes(attr_string):
    attrs = {}
    parts = attr_string.strip().split(";")
    for part in parts:
        if part.strip() == "":
            continue
        key, value = part.strip().split(" ", 1)
        attrs[key] = value.replace('"', '')
    return attrs

with open(gtf_file) as f:
    for line in f:
        if line.startswith("#"):
            continue
        
        fields = line.strip().split("\t")
        if len(fields) < 9:
            continue
        
        chrom = fields[0]
        feature = fields[2]
        start = int(fields[3]) - 1   
        end = int(fields[4])
        strand = fields[6]
        attributes = parse_attributes(fields[8])
        
        if "gene_id" not in attributes or "transcript_id" not in attributes:
            continue
        
        gene_id = attributes["gene_id"]
        transcript_id = attributes["transcript_id"]
        
        if feature == "exon":
            gene_transcripts[gene_id].add(transcript_id)
            exon_key = (chrom, start, end, strand, gene_id)
            exon_transcripts[exon_key].add(transcript_id)

# Jetzt optionale Exons identifizieren
with open(output_file, "w") as out:
    for exon, tx_set in exon_transcripts.items():
        chrom, start, end, strand, gene_id = exon
        
        total_tx = len(gene_transcripts[gene_id])
        exon_tx = len(tx_set)
        
        if exon_tx > 0 and exon_tx < total_tx:
            out.write(f"{chrom}\t{start}\t{end}\t{gene_id}\t0\t{strand}\n")

print("optional_exons.bed wurde erzeugt.")